In [19]:
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer


def retrieve_top_k(
    query,
    collection,
    embed_model,
    top_k=3,
    where=None,
):
    query = query.strip()

    if not query:
        raise ValueError("검색 질문이 비어 있습니다.")

    if top_k < 1:
        raise ValueError("top_k는 1 이상이어야 합니다.")

    document_count = collection.count()

    if document_count == 0:
        return []

    query_embedding = embed_model.encode(
        [query],
        normalize_embeddings=True,
    )

    query_kwargs = {
        "query_embeddings": query_embedding.tolist(),
        "n_results": min(top_k, document_count),
        "include": [
            "documents",
            "metadatas",
            "distances",
        ],
    }

    if where:
        query_kwargs["where"] = where

    raw_results = collection.query(**query_kwargs)

    results = []

    for index, chunk_id in enumerate(raw_results["ids"][0]):
        distance = raw_results["distances"][0][index]

        results.append(
            {
                "rank": index + 1,
                "id": chunk_id,
                "page_content": raw_results["documents"][0][index],
                "metadata": raw_results["metadatas"][0][index],
                "distance": distance,
                "score": 1.0 - distance,
            }
        )

    return results


# 프로젝트 루트 탐색
PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("프로젝트 루트를 찾을 수 없습니다.")

    PROJECT_ROOT = PROJECT_ROOT.parent


# 최종 통합 Chroma DB
CHROMA_PATH = PROJECT_ROOT / "chroma_db"
COLLECTION_NAME = "maplestory_guides"

client = chromadb.PersistentClient(
    path=str(CHROMA_PATH)
)

collection = client.get_collection(
    name=COLLECTION_NAME
)

embed_model = SentenceTransformer(
    "jhgan/ko-sroberta-multitask"
)

print("DB 경로:", CHROMA_PATH)
print("컬렉션:", collection.name)
print("문서 수:", collection.count())


results = retrieve_top_k(
    query="히어로 직업에는 어떤게 있어?",
    collection=collection,
    embed_model=embed_model,
    top_k=5,
)

for result in results:
    print("=" * 80)
    print("순위:", result["rank"])
    print("점수:", round(result["score"], 4))
    print("metadata:", result["metadata"])
    print()
    print(result["page_content"])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3290.99it/s]


DB 경로: C:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-p1-team3\chroma_db
컬렉션: maplestory_guides
문서 수: 3694
순위: 1
점수: 0.5676
metadata: {'board_id': 429467338, 'source': 'guide', 'chunk_index': 303, 'name': '[스킬] 링크 스킬', 'section_title': '성장', 'article_id': 406, 'url': 'https://maplestory.nexon.com/Guide/N23GameInformation/Articles/406'}

모험가 시그너스 기사단 레지스탕스 데몬 영웅 노바 레프 아니마 제로 키네시스 마족 본인의 직업이 어느 분류에 해당하는지 알고싶다면 직업 소개 바로가기 데몬슬레이어 스킬명 데몬스 퓨리 패시브 스킬 설명 대상이 보스 몬스터일 경우 내면에 잠재된 분노를 이끌어 내 더욱 강력한 데미지를 입히고 추가 포스를 흡수한다 효과 Lv 1 보스 몬스터 공격시 10 데미지
순위: 2
점수: 0.5639
metadata: {'board_id': 429467338, 'section_title': '성장', 'article_id': 406, 'name': '[스킬] 링크 스킬', 'source': 'guide', 'chunk_index': 305, 'url': 'https://maplestory.nexon.com/Guide/N23GameInformation/Articles/406'}

2 데미지 10 증가 Lv 3 데미지 15 증가 직업별 링크 스킬 정보 바로가기 직업 분류 모험가 시그너스 기사단 레지스탕스 데몬 영웅 노바 레프 아니마 제로 키네시스 마족 본인의 직업이 어느 분류에 해당하는지 알고싶다면 직업 소개 바로가기 영웅 아란 스킬명 콤보킬 어드밴티지 패시브 스킬 설명 폴암의 정령 마하의 신비한 힘을 통해 콤보 플레이에 대한
순위: 3
점수: 0.5604
metadata: {'sourc